### Tools:

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a json schema).
2.  A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like parakeets, macaws, and cockatiels are known for mimicking human speech. But why do they do that? I remember reading somewhere that parrots are highly intelligent and social animals. Maybe their talking is a way to interact with humans?\n\nWait, but why mimic speech specifically? In the wild, maybe they use vocalizations to communicate with their flock. So if they\'re in captivity with humans, they might imitate human sounds to communicate or bond with their owners. Also, there\'s the concept of vocal learning, which is rare in the animal kingdom. Parrots are part of the few species that can learn to produce sounds they hear. So their ability to talk is a learned behavior through repetition and reinforcement.\n\nI think there\'s also something about their anatomy. They have a syrinx instead of a larynx, which allows the

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at the location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'ybchm8795', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 154, 'total_tokens': 245, 'completion_time': 0.134484389, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.006149785, 'prompt_tokens_details': None, 'queue_time': 0.006186641, 'total_time': 0.140634174}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_ru

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute Tools & Collect Results
for tool_call in ai_msg.tool_calls:
    # Execute the Tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass response back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72 degree Farenhiet and sunny."

The weather in Boston is sunny. Let me know if you need more details!


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location they mentioned. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted. No other parameters are needed. Just the name and the arguments. Alright, that should do it.\n', 'tool_calls': [{'id': 'w8hqnnm24', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 153, 'total_tokens': 265, 'completion_time': 0.184034454, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.006086076, 'prompt_tokens_details': None, 'queue_time': 0.006062239, 'total_time': 0.19012053}, 'model_n